<div style="background: linear-gradient(120deg,#0b1f3a 0%,#173f73 55%,#2a7ab8 100%); color: #ffffff; padding: 26px 30px; border-radius: 14px; font-family: 'Segoe UI', Roboto, Arial, sans-serif;">
<div style="font-size: 30px; font-weight: 700; line-height: 1.2; color: #ffffff;">QCC: the 28.1277 public frontier, re-verified</div>
<div style="font-size: 15px; margin-top: 10px; color: #ffffff;">Google Verifiable Quantum Circuit Compilation &middot; best file of each task &middot; weights refitted here with the QCC Circuit Atlas toolkit &middot; every file re-verified with the unchanged official verifiers</div>
</div>

The best public portfolio at the time of writing is the **28.1277** one of **Pramit Das**
([QCC public frontier 28.1277 verified portfolio](https://www.kaggle.com/code/pramitdas/qcc-public-frontier-28-1277-verified-portfolio)),
whose 28.1277 portfolio is the base of this one; it takes the best file of each task from three different authors: the 44-qubit invsqrt `0af21cb8` of **maoxin123**
([QCC v27 | a 44-qubit invsqrt on the 28.09 frontier](https://www.kaggle.com/code/maoxin123/qcc-v27-a-44-qubit-invsqrt-on-the-28-09-frontier)),
the 890-qubit DQI `66b4f18b` of **Balram Srinivasan**
([VQCC 28.10: GF24 closings in Syed v13 DQI](https://www.kaggle.com/code/balramsrinivasan/vqcc-28-10-gf24-closings-in-syed-v13-dqi)),
and the chemistry weights `703843d3` of **Atakan Aldemir**
([QCC: Terminal Readout and Phase Folds](https://www.kaggle.com/code/atakanaldemir/qcc-terminal-readout-and-phase-folds)), on **Lera**'s chemistry loaders
([`marculera/dfthc-c44-build`](https://www.kaggle.com/datasets/marculera/dfthc-c44-build)). **Koushik Rudra**'s [QCC v54 best verified](https://www.kaggle.com/code/koushikrudra/qcc-v54-best-verified) (28.1215) uses the same invsqrt and chemistry with the previous DQI.
This notebook **takes these public files by SHA-256 (nothing is copied by hand), rebuilds the 18-member `submission.zip` and re-verifies all three tasks with the unchanged official verifiers**.
It also re-runs our chemistry refit from the [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) toolkit (`tools/refit`) as a check.
**Add the inputs (the Atlas dataset, Lera's dataset and the notebooks above) or Copy &amp; Edit this notebook, which keeps them attached.**
If this notebook helps you, an upvote is appreciated.

| Task | File | Built by |
|---|---|---|
| invsqrt | `0af21cb8` (44 qubits) | maoxin123 (on the 28.09 frontier of boristown / Syed Asad Ali) |
| DQI | `66b4f18b` (890 qubits) | Balram Srinivasan (GF24 closings on Syed Asad Ali's v13 decoder; earlier line: hayatohikawa, boristown) |
| chemistry | table, words and loaders from `marculera/dfthc-c44-build`; weights `703843d3` | Lera (marculera); weights: Atakan Aldemir (earlier refits: boristown, jeffemerson) |

## Acknowledgements
Thank you to **Syed Asad Ali**, **jeffemerson** (the merge-cut invsqrt and the chemistry weights), **hayatohikawa** (the DQI improvement), **boristown** (the weights we start from), **Ertu&#287;rul &Ouml;zer**, **boristown** (his [Gap-3 Residue Repair and C43 Native Compiler](https://www.kaggle.com/code/boristown/qcc-gap-3-residue-repair-and-c43-native-compiler), 28.058, carries the same v13 circuits with his own chemistry weights), **Lera (marculera)** (her `NOTICE` is printed below), **boristown**, **maoxin123**, **Atakan Aldemir**,
**Pramit Das**, **Balram Srinivasan**, **Koushik Rudra**, **Heliosli**, **prvsiyan**, **Leo Tang**, **alex chilton**, **manwithacat (James Barlow)**, **Ali Haydar &Ouml;zda&#287;**, **Areeb Ali**, **Kirill Tymchishin**,
**SomeJoint**, **busyaprime**, **yoyobar**, **mrmeows**, **jday96314** and **Oleg Sher**; and Google Quantum AI and Kaggle for the competition, kickmix and the verifiers.

**What is ours here:** the chemistry refit toolkit (dataset), and the independent re-verification of every file with the official verifiers (no new circuit gain of ours in this version). Companion notebook:
[QCC 27.30 | What Works, What Fails and Why](https://www.kaggle.com/code/asteriospanoras/qcc-28-12769-what-works-what-fails-and-why).
#
# *Note on a DQI draw:* boristown's `f425ea20` (a product-chain patch on this DQI) is cheaper at the default salt, but on the public board it drew one failed shot and scored 28.0884, while `66b4f18b` scored 28.1276 (the DQI is sampled, so the board draw is not predictable from the default salt).


<div style="background: #fff7e0; border-left: 6px solid #ffb703; padding: 12px 16px; border-radius: 8px; color: #1b1b1b;">
<b>Getting the submission file.</b> This notebook rebuilds, refits and verifies everything when it runs. To get
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px;">submission.zip</code>: <b>Copy &amp; Edit</b>, set
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px;">WRITE_SUBMISSION = True</code> in the next cell and <b>Run All</b>
(about 50 minutes on CPU). The dataset <b>QCC Circuit Atlas</b> must be attached (it is, after Copy &amp; Edit). Please keep the credits.
</div>

In [ ]:
WRITE_SUBMISSION = False
REFIT_MINUTES = 3
import sys, os, pathlib, zipfile, json, time, subprocess, hashlib, math, csv, io, base64
import numpy as np
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
inp = pathlib.Path("/kaggle/input")
KM = next(p for p in inp.rglob("kickmix") if p.is_dir() and (p.parent / "verifiers").is_dir())
VER = KM.parent / "verifiers"; sys.path.insert(0, str(KM))
sha = lambda b: hashlib.sha256(b).hexdigest()
SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
# Find every public file we need by its SHA-256, whatever notebook carries it.
WANT = {"invsqrt.kmb": "0af21cb8", "dqi.kmb": "66b4f18b"}
CHEM_NPZ = "302e608c"          # marculera/dfthc-c44-build submission.npz (table, words); loaders: its 15 angle_k_*.kmb
got = {}
for z in inp.rglob("submission.zip"):
    try: Z = zipfile.ZipFile(z)
    except Exception: continue
    for n, pre in WANT.items():
        if n in got or n not in Z.namelist(): continue
        b = Z.read(n)
        if sha(b).startswith(pre): got[n] = b
assert set(got) == set(WANT), sorted(got)
for n in WANT: (SUB / n).write_bytes(got[n])
CH = next((p.parent for p in inp.rglob("angle_k_14.kmb") if (p.parent / "submission.npz").is_file()
           and sha((p.parent / "submission.npz").read_bytes()).startswith(CHEM_NPZ)), None)
assert CH, "Attach the dataset marculera/dfthc-c44-build (Add Input)"
for p in CH.iterdir():
    if p.name == "submission.npz" or p.name.startswith("angle_k_"): (SUB / "dfthcsos" / p.name).write_bytes(p.read_bytes())
# Chemistry weights: Atakan Aldemir's refit of Lera's table (npz 703843d3) - the best public weights; our refit continues from them.
_bn = None
for _z in inp.rglob("submission.zip"):
    try: _Z = zipfile.ZipFile(_z)
    except Exception: continue
    if "dfthcsos/submission.npz" in _Z.namelist():
        _b = _Z.read("dfthcsos/submission.npz")
        if sha(_b).startswith("703843d3"): _bn = _b; break
assert _bn, "Attach the notebook pramitdas/qcc-public-frontier-28-1277-verified-portfolio (Add Input)"
(SUB / "dfthcsos/submission.npz").write_bytes(_bn)
NOTICE_TEXT = (CH / "NOTICE").read_text() if (CH / "NOTICE").is_file() else ""
print(NOTICE_TEXT)
log("public files", {n: sha((SUB / n).read_bytes())[:8] for n in WANT}, "chemistry", sha((SUB / "dfthcsos/submission.npz").read_bytes())[:8], "loaders", len(list((SUB / "dfthcsos").glob("angle_k_*"))))

## Our change: refit the chemistry weights here, with the Atlas toolkit
The refit code (our numpy port of prvsiyan's generalised reduced-gradient refit, Apache-2.0, with a full Newton
restoration step and Lera's gauge helpers) lives in the dataset **QCC Circuit Atlas**, folder `tools/refit`, and is
imported from there. It only changes `W` and `H2_I`; the verifier's own math (the competition's `dfthcsos.scoreinfra`)
checks every saved point exactly.

In [ ]:
import glob
ATLAS = next((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/circuits.csv", recursive=True)
              if os.path.isdir(os.path.join(os.path.dirname(p), "tools", "refit"))), None)
assert ATLAS, "Attach the dataset asteriospanoras/qcc-circuit-atlas (Add Input) and run again."
sys.path.insert(0, os.path.join(ATLAS, "tools"))
os.environ["QCC_VERIFIERS"] = str(VER); os.environ["QCC_CACHE"] = "/tmp/refitcache"
import atlas_io as A
from refit import chemeval as ce
from refit.model import Model, split, unsplit
from refit.grg2 import GRG2
ce.setup(verifiers=str(VER))
log("atlas tools from", ATLAS)
# invsqrt f2cdab0b and dqi ccc2ab84 come from the pinned public files above

base = ce.load_npz(str(SUB / "dfthcsos/submission.npz"))
w0 = base["W"].astype(np.float64); wB0 = base["H2_I"].astype(np.float64)
def metrics(ww, bb): return ce.evaluate(base["angle_words"], ww, bb - ww.sum(1), base["tree_edges"])
m0 = metrics(w0, wB0); log("start", {k: m0[k] for k in ("lambda_eff", "error_1", "error_2", "E_gap")})
best = {"leff": m0["lambda_eff"], "W": w0, "H2_I": wB0}
deadline = time.time() + 60 * REFIT_MINUTES
class Stop(Exception): pass
def on_check(xe, okr):
    p, n, hp, hn = unsplit(xe, w0.shape); ww, bb = p - n, hp - hn
    m = metrics(ww, bb)
    ok = m["error_2"] < 0.02 - 1e-9 and m["error_1"] < 0.02 - 1e-9 and m["E_gap"] > 3.914898 + 1e-9 and m["gap_in_domain"] and m["Lambda"] > 49.0
    if ok and m["lambda_eff"] < best["leff"]:
        best.update(leff=m["lambda_eff"], W=ww.copy(), H2_I=bb.copy()); log("better", m["lambda_eff"])
    if time.time() > deadline: raise Stop()
M = Model(base["angle_words"], base["tree_edges"], *w0.shape[:2], cache_key="kaggle")
G = GRG2(M, w0.shape, log=lambda *a: None, m2=1.5e-9, mg=1.5e-9, m1=2e-9)
try: G.run(split(w0, wB0), 100000, on_check=on_check, check_every=50)
except Stop: pass
log("refit lambda_eff", m0["lambda_eff"], "->", best["leff"])
arrays = {k: np.array(base[k]) for k in base}
arrays["W"] = best["W"]; arrays["H2_I"] = best["H2_I"]
with open(SUB / "dfthcsos/submission.npz", "wb") as f: np.savez(f, **arrays)

## Official verification of the exact bytes that go into `submission.zip`

In [ ]:
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"])
log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res

In [ ]:
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}
r = {k: REF[k] / res[k]["score"] for k in REF}
print({k: round(x, 6) for k, x in r.items()}, "GM", round(math.prod(r.values()) ** (1 / 3), 5))
OUT_ZIP = "/kaggle/working/submission.zip" if WRITE_SUBMISSION else "/tmp/submission.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 14, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
json.dump({"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res},
          open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written to", OUT_ZIP)

## Where each file can be found (read from the QCC Circuit Atlas)
The [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) lists every public circuit
file of this competition once, with the public notebooks and datasets that hold the same bytes. The source shown is
one of them (the first in scan order, not necessarily the author: see the credits below for who built each file).
Files newer than the atlas scan (2026-11-12) are marked as such.

In [ ]:
rows = A.circuits()
print(f"{'file':28s} {'sha12':12s} {'found in (one of)':58s} {'owner':22s} {'#sources':>8s}")
for p in sorted(SUB.rglob("*")):
    if not p.is_file(): continue
    h = sha(p.read_bytes())[:12]; r = rows.get(h)
    if r: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {r['first_source_ref'][:58]:58s} {r['author'][:22]:22s} {r['n_sources']:>8s}")
    else: print(f"{str(p.relative_to(SUB)):28s} {h:12s} {'(newer than the atlas scan)':58s}")

## Credits (who did what), with links
- **Portfolio**: [Pramit Das](https://www.kaggle.com/code/pramitdas/qcc-public-frontier-28-1277-verified-portfolio) assembled the 28.1277 combination re-verified here; [alex chilton](https://www.kaggle.com/code/alexchilton/28-09-lb-which-circuits-survive-private-salt) and [boristown](https://www.kaggle.com/code/boristown/qcc-gap-3-residue-repair-and-c43-native-compiler) the 28.09 portfolios before it.
- **invsqrt** `0af21cb8`: [maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-a-44-qubit-invsqrt-on-the-28-09-frontier), on boristown's `f2cdab0b` and [Syed Asad Ali](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook)'s v13 44-qubit circuit and jeffemerson's merge cut (Apache-2.0).
  **DQI** `66b4f18b`: [Balram Srinivasan](https://www.kaggle.com/code/balramsrinivasan/vqcc-28-10-gf24-closings-in-syed-v13-dqi), on Syed Asad Ali's v13 890-qubit decoder, [hayatohikawa](https://www.kaggle.com/code/hayatohikawa/qcc-c8-milestone-public-repro-lb-28-066) (C8 milestone) and boristown's product-chain patch.
  **Chemistry weights** `703843d3`: [Atakan Aldemir](https://www.kaggle.com/code/atakanaldemir/qcc-terminal-readout-and-phase-folds); earlier refits: boristown, jeffemerson, ozertuu.
- **Chemistry**: the table, words and 15 loaders of [Lera (marculera)](https://www.kaggle.com/datasets/marculera/dfthc-c44-build) (Apache-2.0; her `NOTICE` is printed above and the origin is
  `marculera/dfthc-c44-build`), built on the public work her NOTICE lists ([maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-4033-toffoli-coded-word-chemistry), somejoint, Syed Asad Ali, Heliosli,
  James Barlow, Atakan Aldemir and others). Refit method: [prvsiyan](https://www.kaggle.com/datasets/prvsiyan/vqcc-verified-best-of-circuits-1106a) (Apache-2.0), Lera's DF-THC line.
  **Ours:** the refit toolkit in the [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) and the refit run here.
- Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each file keeps its authors' terms; this notebook's code is Apache-2.0.